In [7]:
# Hospital Readmission - Data Preprocessing & Feature Engineering
# Author:HAMOUMA Sihem
# Date: 16/11/2025

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("DATA PREPROCESSING PIPELINE")
print("="*70)

# ============================================================
# STEP 1: LOAD DATA
# ============================================================

print("\n[1/10] Loading raw data...")
df = pd.read_csv(r"C:\Users\lenovo\Desktop\hospital-readmission-prediction\data\raw\diabetic_data.csv")
print(f"✓ Loaded {df.shape[0]} records with {df.shape[1]} features")

# ============================================================
# STEP 2: HANDLE MISSING VALUES
# ============================================================

print("\n[2/10] Handling missing values...")

# Replace '?' with NaN
df = df.replace('?', np.nan)

# Strategy for each column type:
# 1. Drop columns with >40% missing
missing_pct = (df.isnull().sum() / len(df)) * 100
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()
print(f"   Dropping {len(cols_to_drop)} columns with >40% missing: {cols_to_drop}")
df = df.drop(columns=cols_to_drop)

# 2. Fill race with mode
if 'race' in df.columns:
    df['race'].fillna(df['race'].mode()[0], inplace=True)

# 3. Drop remaining rows with missing critical values (small percentage)
initial_rows = len(df)
df = df.dropna()
print(f"✓ Removed {initial_rows - len(df)} rows with remaining missing values")
print(f"✓ Dataset now: {df.shape[0]} records")

# ============================================================
# STEP 3: REMOVE DUPLICATES AND IDENTIFIERS
# ============================================================

print("\n[3/10] Removing duplicates and identifiers...")

# Drop patient and encounter IDs (not features for prediction)
id_columns = ['encounter_id', 'patient_nbr']
df = df.drop(columns=[col for col in id_columns if col in df.columns])

# Remove duplicate patients (keep first encounter)
# Note: In real scenario, might want to aggregate multiple encounters
initial_records = len(df)
# This dataset uses encounter_id, so all are unique
print(f"✓ Data contains unique encounters")

# ============================================================
# STEP 4: TARGET VARIABLE ENGINEERING
# ============================================================

print("\n[4/10] Engineering target variable...")

# Strategy: Focus on early readmission (<30 days) as HIGH RISK
# This is most critical for hospitals to prevent

# Original: NO, >30, <30
# New: 0 (No readmission), 1 (Late readmission), 2 (Early readmission - HIGH RISK)

df['readmitted_binary'] = df['readmitted'].map({
    'NO': 0,
    '>30': 1,
    '<30': 2
})

print("   Target distribution:")
print(df['readmitted_binary'].value_counts())
print("\n   Interpretation:")
print("   0 = No readmission")
print("   1 = Readmitted after 30 days") 
print("   2 = Readmitted within 30 days (HIGH RISK)")

# ============================================================
# STEP 5: FEATURE ENGINEERING - MEDICAL CHANGES
# ============================================================

print("\n[5/10] Creating medication change features...")

# Count total medication changes
medication_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
                   'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
                   'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
                   'miglitol', 'troglitazone', 'tolazamide', 'insulin',
                   'glyburide-metformin', 'glipizide-metformin']

# Create features
df['total_medication_changes'] = 0
df['diabetic_med_prescribed'] = (df['diabetesMed'] == 'Yes').astype(int)

for col in medication_cols:
    if col in df.columns:
        # Count changes (Up, Down, Steady → binary: changed or not)
        df[col + '_changed'] = df[col].isin(['Up', 'Down']).astype(int)
        df['total_medication_changes'] += df[col + '_changed']

print(f"✓ Created medication change features")
print(f"   Average medication changes: {df['total_medication_changes'].mean():.2f}")

# ============================================================
# STEP 6: FEATURE ENGINEERING - DIAGNOSIS GROUPING
# ============================================================

print("\n[6/10] Grouping diagnosis codes...")

# ICD-9 diagnosis code grouping
def categorize_diagnosis(diag_code):
    """Group ICD-9 codes into major disease categories"""
    if pd.isna(diag_code):
        return 'Unknown'
    
    try:
        code = float(diag_code)
        if 390 <= code < 460 or code == 785:
            return 'Circulatory'
        elif 460 <= code < 520 or code == 786:
            return 'Respiratory'
        elif 520 <= code < 580 or code == 787:
            return 'Digestive'
        elif 250 <= code < 251:
            return 'Diabetes'
        elif 800 <= code < 1000:
            return 'Injury'
        elif 710 <= code < 740:
            return 'Musculoskeletal'
        elif 580 <= code < 630 or code == 788:
            return 'Genitourinary'
        elif 140 <= code < 240:
            return 'Neoplasms'
        else:
            return 'Other'
    except:
        return 'Unknown'

# Apply to diagnosis columns
for diag_col in ['diag_1', 'diag_2', 'diag_3']:
    if diag_col in df.columns:
        df[diag_col + '_category'] = df[diag_col].apply(categorize_diagnosis)

print("✓ Diagnosis codes grouped into disease categories")

# ============================================================
# STEP 7: FEATURE ENGINEERING - PATIENT RISK SCORE
# ============================================================

print("\n[7/10] Creating patient risk scores...")

# Create composite risk scores
df['prior_utilization_score'] = (
    df['number_outpatient'] + 
    df['number_emergency'] * 2 +  # Weight emergency higher
    df['number_inpatient'] * 3     # Weight inpatient highest
)

df['comorbidity_score'] = df['number_diagnoses']

df['treatment_intensity_score'] = (
    df['num_lab_procedures'] * 0.1 +
    df['num_procedures'] * 0.5 +
    df['num_medications'] * 0.3
)

print("✓ Created risk scores:")
print(f"   Prior utilization score (mean): {df['prior_utilization_score'].mean():.2f}")
print(f"   Comorbidity score (mean): {df['comorbidity_score'].mean():.2f}")
print(f"   Treatment intensity (mean): {df['treatment_intensity_score'].mean():.2f}")

# ============================================================
# STEP 8: ENCODE CATEGORICAL VARIABLES
# ============================================================

print("\n[8/10] Encoding categorical variables...")

# Age groups - ordinal encoding
age_mapping = {
    '[0-10)': 0, '[10-20)': 1, '[20-30)': 2, '[30-40)': 3,
    '[40-50)': 4, '[50-60)': 5, '[60-70)': 6, '[70-80)': 7,
    '[80-90)': 8, '[90-100)': 9
}
df['age_encoded'] = df['age'].map(age_mapping)

# Gender - binary
df['gender_encoded'] = (df['gender'] == 'Male').astype(int)

# One-hot encode low-cardinality categoricals
categorical_for_onehot = ['race', 'diag_1_category', 'diag_2_category', 'diag_3_category']
categorical_for_onehot = [col for col in categorical_for_onehot if col in df.columns]

df_encoded = pd.get_dummies(df, columns=categorical_for_onehot, prefix=categorical_for_onehot)

print(f"✓ Encoded categorical variables")
print(f"   Dataset shape after encoding: {df_encoded.shape}")

# ============================================================
# STEP 9: FEATURE SELECTION
# ============================================================

print("\n[9/10] Selecting final features...")

# Drop original text columns and redundant features
cols_to_drop_final = ['readmitted', 'age', 'gender', 'race']
cols_to_drop_final += [col for col in medication_cols if col in df_encoded.columns]
cols_to_drop_final += [col for col in df_encoded.columns if '_changed' in col]  # Keep total count only
cols_to_drop_final += ['diag_1', 'diag_2', 'diag_3']  # Keep categories only
cols_to_drop_final += ['diabetesMed', 'change', 'A1Cresult']  # Already captured in features

# Keep only columns that exist
cols_to_drop_final = [col for col in cols_to_drop_final if col in df_encoded.columns]
df_final = df_encoded.drop(columns=cols_to_drop_final)

print(f"✓ Final feature set: {df_final.shape[1] - 1} features")

# Separate features and target
X = df_final.drop('readmitted_binary', axis=1)
y = df_final['readmitted_binary']

print(f"\n   Features shape: {X.shape}")
print(f"   Target shape: {y.shape}")

# ============================================================
# STEP 10: TRAIN-TEST SPLIT & SCALING
# ============================================================

print("\n[10/10] Splitting data and scaling features...")

# Split data (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✓ Train set: {X_train.shape[0]} samples")
print(f"✓ Test set: {X_test.shape[0]} samples")

# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for saving
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print(f"✓ Features scaled (StandardScaler)")

# ============================================================
# SAVE PROCESSED DATA
# ============================================================

print("\n" + "="*70)
print("SAVING PROCESSED DATA")
print("="*70)

# Save processed datasets
X_train_scaled_df.to_csv('../data/processed/X_train.csv', index=False)
X_test_scaled_df.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save scaler for deployment
joblib.dump(scaler, '../models/scaler.pkl')

# Save feature names
feature_names = X_train.columns.tolist()
pd.DataFrame({'feature': feature_names}).to_csv('../data/processed/feature_names.csv', index=False)

print("✓ Saved processed training data: data/processed/X_train.csv")
print("✓ Saved processed test data: data/processed/X_test.csv")
print("✓ Saved scaler: models/scaler.pkl")
print("✓ Saved feature names: data/processed/feature_names.csv")

# ============================================================
# PREPROCESSING SUMMARY
# ============================================================

print("\n" + "="*70)
print("PREPROCESSING SUMMARY")
print("="*70)

summary = f"""
Original Dataset: {df.shape[0]} records, {df.shape[1]} features
Final Dataset: {X_train.shape[0] + X_test.shape[0]} records, {X.shape[1]} features

Data Cleaning:
  ✓ Handled missing values
  ✓ Removed duplicates and identifiers
  ✓ Encoded categorical variables

Feature Engineering:
  ✓ Created medication change features
  ✓ Grouped diagnosis codes into categories
  ✓ Built patient risk scores:
     - Prior utilization score
     - Comorbidity score  
     - Treatment intensity score

Target Variable:
  ✓ 0 = No readmission: {(y == 0).sum()} samples
  ✓ 1 = Late readmission (>30 days): {(y == 1).sum()} samples
  ✓ 2 = Early readmission (<30 days): {(y == 2).sum()} samples

Train-Test Split:
  ✓ Training: {len(y_train)} samples ({len(y_train)/(len(y_train)+len(y_test))*100:.1f}%)
  ✓ Testing: {len(y_test)} samples ({len(y_test)/(len(y_train)+len(y_test))*100:.1f}%)

Next Steps:
  → Build classification models
  → Handle class imbalance (SMOTE)
  → Optimize hyperparameters
  → Evaluate model performance
"""

print(summary)
print("\n✅ Preprocessing Complete! Ready for modeling.")

DATA PREPROCESSING PIPELINE

[1/10] Loading raw data...
✓ Loaded 101766 records with 50 features

[2/10] Handling missing values...
   Dropping 4 columns with >40% missing: ['weight', 'medical_specialty', 'max_glu_serum', 'A1Cresult']
✓ Removed 40977 rows with remaining missing values
✓ Dataset now: 60789 records

[3/10] Removing duplicates and identifiers...
✓ Data contains unique encounters

[4/10] Engineering target variable...
   Target distribution:
readmitted_binary
0    32580
1    21549
2     6660
Name: count, dtype: int64

   Interpretation:
   0 = No readmission
   1 = Readmitted after 30 days
   2 = Readmitted within 30 days (HIGH RISK)

[5/10] Creating medication change features...
✓ Created medication change features
   Average medication changes: 0.34

[6/10] Grouping diagnosis codes...
✓ Diagnosis codes grouped into disease categories

[7/10] Creating patient risk scores...
✓ Created risk scores:
   Prior utilization score (mean): 2.99
   Comorbidity score (mean): 7.67
  

ValueError: could not convert string to float: 'MC'